In [ ]:
# AGI Bench: Proactive/Retroactive Interference
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

**Interference** — when learning new information disrupts old (retroactive) or old information impedes new learning (proactive) — is a fundamental memory phenomenon (Underwood, 1957). This benchmark measures both types using competing rule systems.

### Scoring (v2)
Uses a no-interference control baseline for proper normalization:
- **Retroactive interference** = control_A - post_interference_A (normalized by control)
- **Proactive interference** = control_A - baseline_B (normalized by control)
- **Compartmentalization** = post_interf_A / control_A (retention ratio)
- **Control accuracy** = baseline learning ability

Score = 0.25 × retro_norm + 0.25 × proactive_norm + 0.25 × compartmentalization + 0.25 × control_accuracy


### References
Underwood (1957), Postman (1961), Anderson (2003)


In [ ]:
"""
Novel Rule System Generator for Learning Benchmarks.

Generates procedural rule systems that cannot be in training data.
Each system defines a mapping from inputs to outputs via a chain
of deterministic rules. Difficulty is controlled by:
- Number of rules
- Number of input features
- Rule interaction complexity (independent vs. chained)

Systems are seeded for reproducibility across runs.
"""

import random
import hashlib
from dataclasses import dataclass, field


@dataclass
class RuleSystem:
    """A generated rule system with examples."""
    name: str
    description: str
    rules: list[str]
    examples: list[dict]  # {"input": str, "output": str}
    test_items: list[dict]  # {"input": str, "output": str}
    difficulty: int  # 1-3
    n_rules: int
    domain: str  # "symbol", "language", "number"


def _make_rng(seed: str) -> random.Random:
    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)
    return random.Random(h)


def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a symbol transformation rule system.

    Input: sequence of symbols (e.g., "△ ○ □")
    Rules: transformations (e.g., "△ followed by ○ becomes ★")
    Output: transformed sequence
    """
    rng = _make_rng(seed)

    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
    colors = ["red", "blue", "green", "yellow"]

    if difficulty == 1:
        # Simple 1-to-1 substitution
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            return [mapping.get(s, s) for s in seq]

    elif difficulty == 2:
        # Context-dependent: pairs matter
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            result = []
            i = 0
            while i < len(seq):
                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                    result.extend([pair_rule[2], pair_rule[2]])
                    i += 2
                else:
                    result.append(mapping.get(seq[i], seq[i]))
                    i += 1
            return result

    else:  # difficulty == 3
        # Multi-pass with conditional rules
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]
        cond = src[2]  # If this symbol is present, apply extra rule
        extra_map = {src[3]: dst[3]}

        rules = [
            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()
        ]
        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")
        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")
        rules.append("All other symbols stay the same throughout")

        def apply_rules(seq):
            # Pass 1
            result = [mapping1.get(s, s) for s in seq]
            # Pass 2
            result = [mapping2.get(s, s) for s in result]
            # Conditional
            if cond in seq:  # Check original sequence
                result = [extra_map.get(s, s) for s in result]
            return result

    # Generate examples
    all_items = []
    for _ in range(25):
        length = rng.randint(3, 6)
        seq = [rng.choice(shapes[:5]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    # Deduplicate by input
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_examples = min(15, len(unique_items) - 5)
    examples = unique_items[:n_examples]
    test_items = unique_items[n_examples:n_examples + 5]

    return RuleSystem(
        name=f"SymbolTransform-{seed}",
        description="Apply symbol transformation rules to input sequences",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="symbol",
    )


def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a novel number system / arithmetic.

    Input: expression in the invented system
    Rules: how operators work
    Output: numeric result
    """
    rng = _make_rng(seed)

    op_names = ["grok", "flim", "zorp", "quex", "blix"]
    ops = rng.sample(op_names, 3)

    if difficulty == 1:
        a_op, b_op = ops[0], ops[1]
        a_fn = lambda x, y: x + y + 1
        b_fn = lambda x, y: abs(x - y)
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: absolute difference of x and y",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    elif difficulty == 2:
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        a_fn = lambda x, y: x * 2 + y
        b_fn = lambda x, y: (x + y) % 10
        c_fn = lambda x, y: max(x, y) - min(x, y) + 1
        rules = [
            f"'{a_op}(x, y)' means: double x, then add y",
            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",
            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",
        ]
        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}

    else:
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        a_fn = lambda x, y: x + y + 1
        b_fn = lambda x, y: x * y
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: multiply x and y",
            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    all_items = []
    for _ in range(20):
        if difficulty <= 2:
            op_name = rng.choice(list(op_map.keys()))
            x = rng.randint(1, 9)
            y = rng.randint(1, 9)
            result = op_map[op_name](x, y)
            expr = f"{op_name}({x}, {y})"
        else:
            if rng.random() < 0.5:
                op_name = rng.choice(list(op_map.keys()))
                x = rng.randint(1, 9)
                y = rng.randint(1, 9)
                result = op_map[op_name](x, y)
                expr = f"{op_name}({x}, {y})"
            else:
                inner_op = rng.choice(list(op_map.keys()))
                outer_op = rng.choice(list(op_map.keys()))
                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)
                inner_result = op_map[inner_op](x, y)
                result = op_map[outer_op](inner_result, z)
                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"

        all_items.append({"input": expr, "output": str(result)})

    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_ex = min(12, len(unique_items) - 5)
    examples = unique_items[:n_ex]
    test_items = unique_items[n_ex:n_ex + 5]

    return RuleSystem(
        name=f"NumberSystem-{seed}",
        description="Evaluate expressions using novel arithmetic operators",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="number",
    )


# Pre-generated systems
LEARNING_CURVE_SYSTEMS = [
    generate_symbol_system("lc_sym_easy", difficulty=1),
    generate_symbol_system("lc_sym_med", difficulty=2),
    generate_symbol_system("lc_sym_hard", difficulty=3),
    generate_number_system("lc_num_easy", difficulty=1),
    generate_number_system("lc_num_med", difficulty=2),
    generate_number_system("lc_num_hard", difficulty=3),
]

TRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)
TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)
TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)

INTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)
INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)


In [ ]:
"""
Learning Benchmark 3: Proactive & Retroactive Interference (v2)

Redesigned scoring that discriminates between different interference
strategies using a no-interference control baseline.

Cognitive Science Basis:
- Underwood (1957): Proactive inhibition in retention
- Postman (1961): Retroactive inhibition
- Anderson (2003): Retrieval-induced forgetting

Protocol:
1. Learn A alone → Test A (control_A: no-interference baseline)
2. Learn A then B → Test B (baseline_B)
3. After both: Re-test A (post_interference_A)
4. Compute interference magnitudes relative to control

Score = 0.25 * retro_magnitude_norm + 0.25 * proactive_magnitude_norm
      + 0.25 * compartmentalization + 0.25 * control_accuracy
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import re
import json
# generate_symbol_system defined above


@dataclass
class InterfAnswer:
    answer: str


def normalize_output(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text


def check_output(model_output: str, expected: str) -> bool:
    m = normalize_output(model_output)
    e = normalize_output(expected)
    return e in m or m in e


def test_system(llm, system, context_prefix: str = "", chat_prefix: str = "test") -> float:
    """Test model on a system's test items. Returns accuracy."""
    correct = 0
    rules_text = f"**{system.name}**\n"
    for r in system.rules:
        rules_text += f"- {r}\n"
    examples_text = "\n**Examples:**\n"
    for ex in system.examples[:8]:
        examples_text += f"  {ex['input']} → {ex['output']}\n"

    for ti, test_item in enumerate(system.test_items):
        with kbench.chats.new(f"{chat_prefix}_{ti}"):
            prompt = (
                context_prefix +
                f"\nApply these rules:\n{rules_text}{examples_text}\n"
                f"Input: {test_item['input']}\n\n"
                f"Respond with ONLY: {{\"answer\": \"<output>\"}}"
            )
            try:
                result = llm.prompt(prompt, schema=InterfAnswer)
                answer = result.answer
            except Exception:
                raw = llm.prompt(prompt)
                try:
                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                    answer = str(parsed.get("answer", raw))
                except Exception:
                    answer = raw

            if check_output(answer, test_item["output"]):
                correct += 1

    return correct / len(system.test_items) if system.test_items else 0


# Generate two similar systems that should interfere
SYSTEM_A = generate_symbol_system("interf_alpha_v2", difficulty=2)
SYSTEM_B = generate_symbol_system("interf_beta_v2", difficulty=2)


def compute_interference_score(control_A, baseline_B, post_interf_A):
    """Compute interference metrics and composite score."""
    retro_raw = max(0.0, control_A - post_interf_A)
    retro_norm = retro_raw / control_A if control_A > 0 else 0.0
    proactive_raw = max(0.0, control_A - baseline_B)
    proactive_norm = proactive_raw / control_A if control_A > 0 else 0.0
    compartment = post_interf_A / control_A if control_A > 0 else 0.0
    compartment = min(1.0, compartment)
    score = round(
        0.25 * retro_norm + 0.25 * proactive_norm
        + 0.25 * compartment + 0.25 * control_A, 4
    )
    return {
        'retro_raw': retro_raw, 'retro_norm': retro_norm,
        'proactive_raw': proactive_raw, 'proactive_norm': proactive_norm,
        'compartmentalization': compartment, 'control_A': control_A,
        'composite_score': max(0.0, min(1.0, score)),
    }


@kbench.task(name="learning_interference")
def learning_interference(llm) -> float:
    """
    Proactive & Retroactive Interference Benchmark (v2).
    """
    # Phase 1: Control — Learn A alone
    control_A = test_system(llm, SYSTEM_A, chat_prefix="control_A")

    # Phase 2: Learn B after A context
    a_context = (
        f"You previously learned system {SYSTEM_A.name}. "
        f"Now learn a NEW but similar system.\n"
    )
    baseline_B = test_system(llm, SYSTEM_B, context_prefix=a_context, chat_prefix="phase2_B")

    # Phase 3: Re-test A after learning B
    b_context = (
        f"You recently learned two similar systems: "
        f"{SYSTEM_A.name} and {SYSTEM_B.name}. "
        f"Now recall the FIRST system ({SYSTEM_A.name}) specifically. "
        f"Ignore the second system.\n"
    )
    post_interf_A = test_system(llm, SYSTEM_A, context_prefix=b_context,
                                 chat_prefix="phase3_retest_A")

    metrics = compute_interference_score(control_A, baseline_B, post_interf_A)
    score = metrics['composite_score']

    print(f"\n{'='*60}")
    print(f"PROACTIVE & RETROACTIVE INTERFERENCE RESULTS (v2)")
    print(f"{'='*60}")
    print(f"System A: {SYSTEM_A.name} ({len(SYSTEM_A.rules)} rules)")
    print(f"System B: {SYSTEM_B.name} ({len(SYSTEM_B.rules)} rules)")
    print(f"\n--- Phase Results ---")
    print(f"Control A (no interference): {control_A:.2%}")
    print(f"Baseline B (after A):        {baseline_B:.2%}")
    print(f"Post-interference A:         {post_interf_A:.2%}")
    print(f"\n--- Interference Metrics ---")
    print(f"Retroactive (normalized):   {metrics['retro_norm']:.4f}")
    print(f"Proactive (normalized):     {metrics['proactive_norm']:.4f}")
    print(f"Compartmentalization:       {metrics['compartmentalization']:.4f}")
    print(f"\nComposite score: {score:.4f}")

    return score


# Run
learning_interference.run(llm=kbench.llm)
